# Dusk versus Omnipair v1

## tl;dr
5× centered depth saves 80% of capital for the same local depth but multiplies local gross LVR per dollar by five. At $1m and 100% annual volatility: $342/day versus $1,712/day. These are local benchmarks, not measured returns. Market-wide dynamic-fee effectiveness remains uncalibrated.

## Context & Methods
Companion to the report.

### Key Assumptions
Balanced center price=1; ordinary reserve NAV is capital; full cash, no active hLP or public borrowing, frozen center. Real-valued source-derived nested curve; no atom parity claim. Fixed fee experiments do not implement the dynamic schedule.

Sources: Dusk curve.rs at 26bc75b217a96a6241b82b05cd52c1b587272dc1; v1 swap.rs at 84a8ef7a2a3a81e750435b86d7191bf43413d6a4; https://arxiv.org/html/2208.06046v5 and https://arxiv.org/abs/2305.14604. Local source hashes: source-notes.json.

In [1]:
from pathlib import Path
import sys, json
analysis_dir = Path.cwd()
if not (analysis_dir / 'model.py').exists():
    analysis_dir = analysis_dir / 'analysis' / 'capital-efficiency'
sys.path.insert(0, str(analysis_dir))
from model import Curve, build_results, validate_model


## Data
Synthetic parameter grid. Change this cell to rerun the model; regenerate the report separately if assumptions change.

In [2]:
parameters = dict(capital=1_000_000, annual_volatility=1.0, core_bps=100, fade_bps=400, fee_bps=30, lp_fee_share=0.8)
results = build_results(**parameters)
print(json.dumps(parameters, indent=2))

{
  "capital": 1000000,
  "annual_volatility": 1.0,
  "core_bps": 100,
  "fade_bps": 400,
  "fee_bps": 30,
  "lp_fee_share": 0.8
}


## Results
### Centered capital and gross local LVR

In [3]:
for row in results['scenarios']:
    print(f"A={row['amplification']:>2}: gross LVR ${row['gross_lvr_usd_day']:,.2f}/day; equal-peak capital ${row['same_peak_depth_capital_usd']:,.0f}; tail {row['tail_depth_vs_v1']:.3f}x")

A= 1: gross LVR $342.47/day; equal-peak capital $1,000,000; tail 1.000x
A= 2: gross LVR $684.93/day; equal-peak capital $500,000; tail 0.985x
A= 5: gross LVR $1,712.33/day; equal-peak capital $200,000; tail 0.941x
A=10: gross LVR $3,424.66/day; equal-peak capital $100,000; tail 0.867x


### Same-size execution and toxic jump

In [4]:
for row in results['executions']:
    if row['gross_quote'] == 10000 and row['fee_bps'] == parameters['fee_bps']:
        print(f"A={row['amplification']:>2}: $10k buy cost {row['average_cost_bps']:.2f} bps")
for row in results['shocks']:
    if row['jump'] == 0.01 and row['fee_bps'] == parameters['fee_bps']:
        print(f"A={row['amplification']:>2}: +1% jump LP shortfall ${row['lp_net_shortfall']:.2f}")

A= 1: $10k buy cost 230.09 bps
A= 2: $10k buy cost 138.53 bps
A= 5: $10k buy cost 70.09 bps
A=10: $10k buy cost 50.09 bps
A= 1: +1% jump LP shortfall $7.12
A= 2: +1% jump LP shortfall $14.23
A= 5: +1% jump LP shortfall $35.58
A=10: +1% jump LP shortfall $71.17


### Validation
Checks centered NAV, derivative/gamma, equal-depth LVR, reciprocal symmetry, boundary continuity, fee conservation, CPMM closed forms and infeasible width/amplification rejection.

In [5]:
checks = validate_model()
print(chr(10).join(checks))

A=1: centered capital, gamma, equal-depth LVR, reciprocal symmetry, boundary continuity, fee conservation
A=2: centered capital, gamma, equal-depth LVR, reciprocal symmetry, boundary continuity, fee conservation
A=5: centered capital, gamma, equal-depth LVR, reciprocal symmetry, boundary continuity, fee conservation
A=10: centered capital, gamma, equal-depth LVR, reciprocal symmetry, boundary continuity, fee conservation
CPMM exact swap formula
CPMM fee-free jump arbitrage closed form
Infeasible wide/high-amplification rejection


### Dynamic fee extension
The current input-denominated gross-path distance and volatility formulas are modeled in dynamic_fees.py. Example coefficients: distance=10, volatility=1; base=30bps; each surcharge cap=200bps; half-life=60s; protocol takes 20% of base only. Prior accumulators 0/0.005/0.01 are conditional states, not annualized volatility. Center fixed, no active hLP, cash restrictions or compounding. Real-valued arithmetic; not integer program parity.

In [6]:
from dynamic_fees import experiments, validate as validate_dynamic
dynamic = experiments()
for row in dynamic['dynamic_shocks']:
    if row['external_jump'] == 0.01:
        print(f"{row['profile']}: loss distributed ${row['lp_loss_distributed_usd']:.2f}; retained ${row['lp_loss_retained_usd']:.2f}")
print('Dynamic checks:', len(validate_dynamic()))

v1 base only: loss distributed $7.12; retained $7.12
5x base only: loss distributed $35.58; retained $35.58
5x dynamic, quiet: loss distributed $21.20; retained $26.23
5x dynamic, warm: loss distributed $3.03; retained $12.35
5x dynamic, hot: loss distributed $0.00; retained $0.00
Dynamic checks: 10


### Capital after paying dynamic fees
This is a separate finite-size execution experiment: resize capital to match v1's $10k buy at 230.09bps all-in. The shock losses above continue to use equal $1m capital.

In [7]:
for row in dynamic['dynamic_execution']:
    print(f"{row['profile']}: cost {row['average_cost_bps']:.2f}bps; matched-execution capital ${row['capital_for_v1_execution_usd']:,.0f}; saving {row['capital_saved_same_execution']:.1%}")

v1 base only: cost 230.09bps; matched-execution capital $1,000,000; saving 0.0%
5x base only: cost 70.09bps; matched-execution capital $261,446; saving 73.9%
5x dynamic, quiet: cost 122.50bps; matched-execution capital $624,195; saving 37.6%
5x dynamic, warm: cost 172.39bps; matched-execution capital $737,991; saving 26.2%
5x dynamic, hot: cost 222.28bps; matched-execution capital $949,256; saving 5.1%


## Takeaways
1. Choose depth targets at actual order size and fee, not peak multiplier alone.
2. At A=5, residual toxic-cost factor must fall by more than 80% versus v1 to reduce net toxic cost per capital at the center. Organic fee revenue can change net profitability.
3. Replay the same external opportunities with each engine's actual dynamic fees, center policy, cash constraints and hLP settlement before selecting parameters.

Execution note: cells were executed sequentially in Python with captured outputs; a Jupyter kernel and nbformat validator were unavailable. Rerun in Jupyter to certify that separate execution surface.

## Fixed-fee concentration sweep
Twelve Dusk concentrations; fees and widths fixed. See CONCENTRATION_SWEEP.md. Cells below independently recompute execution cases and sample trading paths from either the repository root or analysis directory.

In [8]:
from pathlib import Path
import sys, json
root=Path.cwd()
if not (root/'concentration_sweep.py').exists():
    root=root/'analysis'/'capital-efficiency'
sys.path.insert(0,str(root))
from concentration_sweep import event_experiments, replay, path_inputs
events=event_experiments()
for row in events['concentration_execution']:
    if row['state']=='warm' and row['notional_usd']==10000:
        print(row['profile'],'required capital',round(row['matched_capital_usd']),'saved',round(row['capital_saved_fraction']*100,2))

Omnipair v1 required capital 1000000 saved 0.0
Dusk 1.25x required capital 1362855 saved -36.29
Dusk 1.5x required capital 1225950 saved -22.6
Dusk 2x required capital 1049053 saved -4.91
Dusk 3x required capital 870277 saved 12.97
Dusk 4x required capital 784711 saved 21.53
Dusk 5x required capital 737991 saved 26.2
Dusk 7.5x required capital 684652 saved 31.53
Dusk 10x required capital 659426 saved 34.06
Dusk 15x required capital 635029 saved 36.5
Dusk 20x required capital 623146 saved 37.69
Dusk 30x required capital 611474 saved 38.85
Dusk 50x required capital 602289 saved 39.77


In [9]:
for a in [1,3,5,10]:
    row=replay(a,path_inputs()['choppy_1pct'],2000)
    assert abs(row['net_trading_usd']-row['ledger_net_trading_usd'])<1e-5
    print(row['profile'],'net trading',round(row['net_trading_usd'],2),'fills',row['customer_fills'])

Omnipair v1 net trading 348.0 fills 60
Dusk 3x net trading 412.4 fills 44
Dusk 5x net trading 478.72 fills 51
Dusk 10x net trading 494.23 fills 58


### Interpretation
Fixed parameters do not mean fixed realized fees. No single amplification wins across paths and order assumptions. The warm $1k fee floor cannot be solved with concentration. Cells executed sequentially in Python with captured outputs; this does not certify a Jupyter kernel or on-chain program parity.

## Historical-price shortlist with modeled demand
Prices and prior-day clip sizes are observed; Dusk arrivals, direction and customer cost budgets are assumptions. See HISTORICAL_SHORTLIST.md and source checks in market-data/manifest.json.

In [10]:
from pathlib import Path
import json,sys
root=Path.cwd()
if not (root/'shortlist_replay.py').exists():root=root/'analysis'/'capital-efficiency'
sys.path.insert(0,str(root))
results=json.loads((root/'shortlist_results.json').read_text())
quality=json.loads((root/'shortlist_data_quality.json').read_text())
print(quality['calibration'])
for r in results['summary']:
    if r['experiment']=='primary' and r['demand_target_quote']==1000000 and r['tolerance_median_bps']==100:
        print(r['window'],r['amplification'],round(r['mean_net_trading_quote'],2))

{'agg_trade_clips': 238579, 'first_utc': '2025-01-01T00:00:00.707766+00:00', 'last_utc': '2025-01-01T23:59:57.278264+00:00', 'mean_clip_quote': 1326.8495203139842, 'min_clip_quote': 0.18942, 'max_clip_quote': 283278.31983000005, 'p10_clip_quote': 12.576300000000002, 'p50_clip_quote': 208.97226, 'p90_clip_quote': 3104.32, 'p95_clip_quote': 5848.5632, 'p99_clip_quote': 18190.216, 'observed_aggressor_buy_fraction': 0.5457269918978619, 'sampling': 'Full empirical notional distribution, no trimming; synthetic direction is independently 50/50, not inferred organic classification'}
quiet_p10 1 1140.02
quiet_p10 3 1938.15
quiet_p10 5 2317.98
quiet_p10 10 2402.7
median_p50 1 1214.62
median_p50 3 2036.49
median_p50 5 2370.43
median_p50 10 2614.37
volatile_p90 1 1277.62
volatile_p90 3 1698.61
volatile_p90 5 1875.62
volatile_p90 10 2148.58
extreme_p99 1 1188.67
extreme_p99 3 1832.42
extreme_p99 5 2118.01
extreme_p99 10 2555.14
largest_hourly_move 1 1086.57
largest_hourly_move 3 1298.33
largest_hou

In [11]:
import shortlist_replay as replay
sizes,windows,_=replay.prepare_data()
replay.init_worker(sizes,windows)
case=next(c for c in replay.cases() if c['experiment']=='primary' and c['window_index']==3 and c['amplification']==10 and c['demand_target_quote']==1000000 and c['tolerance_median_bps']==100 and c['seed']==101)
r=replay.run_case(case)
assert abs(r['net_trading_quote']-r['endpoint_ledger_net_trading_quote'])<1e-4
print('Recomputed case:',r['window'],'A=',r['amplification'],'net trading=',r['net_trading_quote'],'fills=',r['fills'])

Recomputed case: extreme_p99 A= 10 net trading= 2617.189937162204 fills= 363


### Limits
These are five independent, centered one-hour episodes, not continuous January performance. Demand is a scenario model, not observed Dusk order flow. No annualization, unhedged LP-return claim, recentering or automatic concentration policy. Cells executed sequentially in Python, not certified in a Jupyter kernel.

## Adaptive concentration
Causal controller experiment with reserve-preserving changes, repricing losses, frozen training selection, and explicitly exploratory cost-gated evaluation. Prices are historical; demand is modeled. Governance admission is relaxed for economic research; no program behavior changed.

In [12]:
from pathlib import Path
import json
root=Path.cwd()
if not (root/'adaptive_results.json').exists(): root=root/'analysis'/'capital-efficiency'
data=json.loads((root/'adaptive_results.json').read_text())
gated=json.loads((root/'adaptive_gated_results.json').read_text())
print('Frozen selection:',data['selected'])
for r in data['summary']+gated['summary']:
    if r['split']=='evaluation' and r['mode']=='all':
        print(r['policy'], 'net',round(r['mean_net_trading_quote'],2),'switch loss',round(r['mean_switch_loss_quote'],2),'drawdown',round(r['mean_max_trading_drawdown_quote'],2))

Frozen selection: {'fixed': 'fixed_10', 'coverage': 'coverage_150', 'flow': 'flow_050', 'volatility': 'volatility_100'}
fixed_1 net 3051.47 switch loss 0.0 drawdown 134.46
fixed_3 net 3359.82 switch loss 0.0 drawdown 14.56
fixed_5 net 3985.54 switch loss 0.0 drawdown 19.96
fixed_10 net 4599.64 switch loss 0.0 drawdown 33.47
coverage_150 net 1818.54 switch loss 2676.2 drawdown 2259.53
flow_050 net 3634.53 switch loss 89.06 drawdown 53.82
volatility_100 net 3318.65 switch loss 824.53 drawdown 705.2
coverage_cost_gate net 4509.49 switch loss 17.43 drawdown 24.56


In [13]:
import sqlite3
conn=sqlite3.connect(':memory:');conn.row_factory=sqlite3.Row
verified=[dict(r) for r in conn.execute((root/'adaptive_verification.sql').read_text())]
conn.close()
for r in verified:
    source=next(v for v in data['summary']+gated['summary'] if all(v[k]==r[k] for k in ['split','mode','policy']))
    assert abs(source['mean_net_trading_quote']-r['mean_net_trading'])<1e-6
print('Independent SQLite means reconciled:',len(verified))
print(json.loads((root/'adaptive_extra_validation.json').read_text())['dense_adaptive_hour'])

Independent SQLite means reconciled: 100
{'policy': 'coverage_150', 'net_trading_quote': 2129.1873645622004, 'dense_net_trading_quote': 2129.186672755398, 'fills': 379, 'switches': 1}


### Interpretation
Mean profit favors fixed10× on this designed grid. The exploratory cost gate trades about2% mean profit for lower sample drawdown; it needs fresh-date validation and an explicit risk objective. Neither minute-scale authority nor protected-depth admission is implemented. The notebook cells are captured sequential Python executions, not Jupyter kernel certification.